**PART 1**

In [2]:
# imports

import json
import os
import re
import subprocess
import time
from pathlib import Path
import pandas as pd
import requests
from huggingface_hub import snapshot_download

data_dir = Path.cwd() / "data"
data_path = data_dir / "data_new.jsonl"

/Users/randy.rakotondrazafy/LOG6307E Data Minig/Replication_Project_LOG6307E/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import data from huggingface if not already downloaded

data_dir.mkdir(parents=True, exist_ok=True)

required_files = {"pull_request.parquet", "human_pull_request.parquet"}
existing_files = {p.name for p in data_dir.glob("*.parquet")}

if not required_files.issubset(existing_files):
    snapshot_download(
        "hao-li/AIDev",
        repo_type="dataset",
        revision="v3",
        local_dir=data_dir,
        allow_patterns=list(required_files),
    )

    for path in data_dir.rglob("*.parquet"):
        if path.name in required_files and path.parent != data_dir:
            path.rename(data_dir / path.name)

Fetching 2 files: 100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


In [3]:
# preare data to github mining

ai_pr = pd.read_parquet(data_dir / "pull_request.parquet")
hu_pr = pd.read_parquet(data_dir / "human_pull_request.parquet")

columns = ["id", "number", "title", "created_at", "repo_url", "html_url", "agent", "group"]
ai_pr["group"] = "ai"
hu_pr["group"] = "human"

frame = pd.concat([ai_pr[columns], hu_pr[columns]], ignore_index=True)
frame = frame.drop_duplicates(subset="id")
frame = pd.concat([frame[frame["group"] == "ai"].sample(frac=0.5, random_state=42), frame[frame["group"] == "human"]], ignore_index=True)

repositories = frame.html_url.str.extract(r"github\.com/([^/]+)/([^/]+)/pull/\d+")
frame["owner"] = repositories[0]
frame["name"] = repositories[1]
frame = frame.dropna(subset=["owner", "name"])

In [4]:
# mine the data from GitHub if not already mined

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
batch = 15

pr = """
p{i}: repository(owner:"{owner}", name:"{name}") {{
  pullRequest(number:{number}) {{
    number title body state isDraft createdAt closedAt mergedAt
    author {{ login __typename }}
    mergedBy {{ login }}
    labels(first:20) {{ nodes {{ name }} }}
    commits(first: 1) {{ totalCount }}
    reopened: timelineItems(first:1, itemTypes:[REOPENED_EVENT]) {{ totalCount }}

    comments(first:100) {{
      totalCount
      nodes {{
        author {{ login __typename }}
        createdAt
        body
      }}
    }}

    reviews(first:50) {{
      totalCount
      nodes {{
        author {{ login __typename }}
        state
        submittedAt
        body

        comments(first:50) {{
          totalCount
          nodes {{
            author {{ login __typename }}
            createdAt
            body
          }}
        }}
      }}
    }}
  }}
}}
"""


def mine():
    rows = list(frame.itertuples(index=False))
    total = len(rows)
    total_batches = (total + batch - 1) // batch

    print(f"Total PRs: {total}")
    print(f"Total batches: {total_batches}")

    with data_path.open("w", encoding="utf-8") as f:

        for start in range(0, total, batch):
            chunk = rows[start:start + batch]
            batch_number = start // batch + 1

            print(f"\nBatch {batch_number}/{total_batches} "
                  f"({start + 1}-{min(start + batch, total)} / {total})")

            try:
                parts = [
                    pr.format(
                        i=i,
                        owner=r.owner,
                        name=r.name,
                        number=int(r.number)
                    )
                    for i, r in enumerate(chunk)
                ]

                response = requests.post(
                    "https://api.github.com/graphql",
                    json={
                        "query": "query {\n" + "\n".join(parts) + "\n}"
                    },
                    headers={
                        "Authorization": f"bearer {GITHUB_TOKEN}"
                    },
                    timeout=60
                )

                print(f"HTTP status: {response.status_code}")

                if response.status_code != 200:
                    print("GitHub error:")
                    print(response.text[:1000])
                    continue

                result = response.json()

                # GraphQL errors can happen even with HTTP 200
                if "errors" in result:
                    print("GraphQL errors:")
                    print(json.dumps(result["errors"], indent=2)[:2000])

                found = 0

                for i, r in enumerate(chunk):
                    key = f"p{i}"

                    pull_request = (
                        result.get("data", {}).get(key, {}) or {}
                    ).get("pullRequest")

                    if pull_request is not None:
                        found += 1

                    f.write(json.dumps({
                        "pr_id": int(r.id),
                        "group": r.group,
                        "agent": r.agent,
                        "repo": f"{r.owner}/{r.name}",
                        "number": int(r.number),
                        "found": pull_request is not None,
                        "pr": pull_request
                    }) + "\n")

                f.flush()

                print(f"Found: {found}/{len(chunk)}")

            except Exception as e:
                print(f"ERROR in batch {batch_number}: {type(e).__name__}: {e}")

    print("\nMining finished.")
if not data_path.exists():
  mine()

Total PRs: 23416
Total batches: 1562

Batch 1/1562 (1-15 / 23416)
HTTP status: 200
GraphQL errors:
[
  {
    "type": "NOT_FOUND",
    "path": [
      "p10"
    ],
    "locations": [
      {
        "line": 403,
        "column": 1
      }
    ],
    "message": "Could not resolve to a Repository with the name 'antiwork/flexile'."
  }
]
Found: 14/15

Batch 2/1562 (16-30 / 23416)
HTTP status: 200
Found: 15/15

Batch 3/1562 (31-45 / 23416)
HTTP status: 200
GraphQL errors:
[
  {
    "type": "NOT_FOUND",
    "path": [
      "p10"
    ],
    "locations": [
      {
        "line": 403,
        "column": 1
      }
    ],
    "message": "Could not resolve to a Repository with the name 'antiwork/helper'."
  }
]
Found: 14/15

Batch 4/1562 (46-60 / 23416)
HTTP status: 200
Found: 15/15

Batch 5/1562 (61-75 / 23416)
HTTP status: 200
Found: 15/15

Batch 6/1562 (76-90 / 23416)
HTTP status: 200
GraphQL errors:
[
  {
    "type": "NOT_FOUND",
    "path": [
      "p3"
    ],
    "locations": [
      {
    

In [ ]:
import requests
from datetime import datetime, timezone

response = requests.post(
    "https://api.github.com/graphql",
    json={"query": "query { rateLimit { remaining resetAt } }"},
    headers={"Authorization": f"bearer {GITHUB_TOKEN}"}
)

rate = response.json()["data"]["rateLimit"]
reset = datetime.fromisoformat(rate["resetAt"].replace("Z", "+00:00"))
now = datetime.now(timezone.utc)

print(f"Remaining: {rate['remaining']}")
print(f"Resets at: {reset.astimezone().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Time remaining: {reset - now}")

**PART 2**

In [3]:
# Remove PRs that were not successfully retrieved

with data_path.open("r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

data = [r for r in data if r.get("found") and r.get("pr")]

print(f"Number of PRs: {len(data)}")
print(pd.Series([r["group"] for r in data]).value_counts().to_string())

Number of PRs: 13475
ai       7438
human    6037


**ADD BY ME**

In [4]:
# Remove duplicate mined PRs and report missing/retrieved records.
mined_rows = len(data)
seen = set()
deduped_data = []

for r in data:
    key = (r.get("repo"), r.get("number"))
    if key in seen:
        continue
    seen.add(key)
    deduped_data.append(r)

mined_duplicate_rows_removed = len(data) - len(deduped_data)
data = deduped_data

print(f"Mined rows: {mined_rows}")
print(f"Duplicate rows removed: {mined_duplicate_rows_removed}")
print(f"Unique retrieved PRs: {len(data)}")
print("\nGroups after cleaning:")
print(pd.Series([r["group"] for r in data]).value_counts().to_string())

Mined rows: 13475
Duplicate rows removed: 10
Unique retrieved PRs: 13465

Groups after cleaning:
ai       7438
human    6027


In [5]:
# Function to identify bots from human and AI agents

bot_suffix = re.compile(r"\[bot\]$", re.IGNORECASE)

common_bots = re.compile(
    r"(?i)^(dependabot|renovate|semantic-release|azure-sdk|calcom-bot|"
    r"github-actions|allcontributors|imgbot|snyk-bot|codecov|netlify|vercel|"
    r"robobun|claassistant|codecov-commenter|vdaas-ci|autogpt-agent|coveralls|"
    r".+-ci|.+-commenter|.+-service\d*|.+-engineering-service\d*|.+-bot\d*)$"
)

def is_bot(login, typename):
    if not isinstance(login, str) or not login:
        return True

    if str(typename).lower() == "bot":
        return True

    return (
        bool(bot_suffix.search(login))
        or bool(common_bots.match(bot_suffix.sub("", login).strip().lower()))
    )

In [6]:
bot_pr_ids = set()
clean_data = []

# Remove bot PRs
for r in data:
    pr = r.get("pr")
    if not pr:
        continue

    author = pr.get("author") or {}

    if is_bot(author.get("login"), author.get("__typename")):
        bot_pr_ids.add(r["pr_id"])

for r in data:
    if r["pr_id"] in bot_pr_ids:
        continue

    pr = r.get("pr")
    if not pr:
        continue

    # Remove bot comments

    comments = pr.get("comments", {}).get("nodes", [])
    pr["comments"]["nodes"] = [c for c in comments if not is_bot((c.get("author") or {}).get("login"), 
        (c.get("author") or {}).get("__typename"))]

    reviews = pr.get("reviews", {}).get("nodes", [])
    clean_reviews = []

    for review in reviews:
        author = review.get("author") or {}

        # Remove bot reviews
        
        if is_bot(
            author.get("login"),
            author.get("__typename")
        ):
            continue

        # Remove bot inline comments

        comments = review.get("comments", {}).get("nodes", [])
        review["comments"]["nodes"] = [c for c in comments if not is_bot((c.get("author") or {}).get("login"),
            (c.get("author") or {}).get("__typename"))]

        clean_reviews.append(review)

    pr["reviews"]["nodes"] = clean_reviews
    clean_data.append(r)

data = clean_data

print(f"Bot PRs removed: {len(bot_pr_ids)}")

Bot PRs removed: 3001


In [7]:
# Text cleaning functions

fence_re = re.compile(r"```.*?```", re.S)
inline_code_re = re.compile(r"`[^`\n]+`")
html_comment_re = re.compile(r"<!--.*?-->", re.S)
html_tag_re = re.compile(r"<[^>]+>")
quote_line_re = re.compile(r"^\s*>.*$", re.M)
url_re = re.compile(r"https?://\S+")
img_re = re.compile(r"!\[[^\]]*\]\([^)]*\)")


def strip_markdown(body):
    if not body:
        return ""

    text = html_comment_re.sub(" ", body)
    text = fence_re.sub(" ", text)
    text = img_re.sub(" ", text)
    text = inline_code_re.sub(" ", text)
    text = quote_line_re.sub(" ", text)
    text = html_tag_re.sub(" ", text)
    text = url_re.sub(" ", text)

    return re.sub(r"\s+", " ", text).strip()


def add_text_fields(event):
    event["text"] = strip_markdown(event.get("body", ""))
    event["n_words"] = len(event["text"].split())


for r in data:
    pr = r.get("pr")
    if not pr:
        continue

    for comment in pr.get("comments", {}).get("nodes", []):
        add_text_fields(comment)

    for review in pr.get("reviews", {}).get("nodes", []):
        add_text_fields(review)

        for comment in review.get("comments", {}).get("nodes", []):
            add_text_fields(comment)

In [10]:
# Define lexicon for clarification language

clarify_re = re.compile("|".join([
    r"\b(?:could|can|would)\s+you\s+(?:please\s+)?(?:explain|clarify|elaborate|confirm)",
    r"\bwhy\s+(?:is|are|was|were|does|did|do|not)\b",
    r"\bwhat\s+(?:is|are|was|does|do|did)\s+the\b",
    r"\bclarif(?:y|ication)\b",
    r"\bnot\s+sure\s+(?:what|why|how|if|that)\b",
    r"\bcan\s+you\s+explain\b",
    r"\bwhat'?s\s+the\s+(?:reason|purpose|point|rationale)\b",
    r"\bany\s+reason\s+(?:why|for)\b",
    r"\bdo\s+we\s+(?:need|want)\b",
    r"\bis\s+(?:this|that|it)\s+(?:intentional|intended|necessary|correct)\b"
]), re.I)


# Define lexicon for approval language

approve_re = re.compile("|".join([
    r"\blgtm\b",
    r"\bsgtm\b",
    r"\blooks?\s+good\b",
    r"\bship\s+it\b",
    r"\bapprov(?:e|ed|ing)\b",
    r"\bnice\s+(?:work|job|catch)\b",
    r"\bgreat\s+(?:work|job|catch)\b",
    r"\bmerging\b",
    r"\bwill\s+merge\b",
    r"\bthanks?\s+for\s+(?:the\s+)?(?:fix|patch|pr|work)\b",
    r"\bperfect\b"
]), re.I)


# Define lexicon for rejection language

reject_re = re.compile("|".join([
    r"\bplease\s+(?:fix|revert|address|update|remove|change)\b",
    r"\bblocking\b",
    r"\bdo(?:\s+not|n'?t)\s+merge\b",
    r"\bneeds?\s+(?:work|changes|fixing)\b",
    r"\brevert(?:ing)?\s+this\b",
    r"\bclosing\s+(?:this|in\s+favou?r)\b",
    r"\bthis\s+is\s+wrong\b",
    r"\bwon'?t\s+(?:work|merge)\b",
    r"\bincorrect\b",
    r"\bbreaks?\s+(?:the\s+)?(?:build|tests?)\b",
    r"\bregression\b",
    r"\bplease\s+don'?t\b",
    r"\bnot\s+(?:correct|right)\b"
]), re.I)


def add_language_flags(event):
    text = event.get("text", "")
    event["is_clarification"] = bool(clarify_re.search(text))
    event["has_approval_lang"] = bool(approve_re.search(text))
    event["has_rejection_lang"] = bool(reject_re.search(text))


for r in data:
    pr = r.get("pr")
    if not pr:
        continue

    for comment in pr.get("comments", {}).get("nodes", []):
        add_language_flags(comment)

    for review in pr.get("reviews", {}).get("nodes", []):
        add_language_flags(review)

        for comment in review.get("comments", {}).get("nodes", []):
            add_language_flags(comment)

**ADD BY ME**

In [8]:
# Handle missing values and standardize fields used by RQ1.

for r in data:
    pr = r["pr"]

    # Empty text is treated as missing/empty rather than dropping the PR.
    pr["title"] = (pr.get("title") or "").strip()
    pr["body"] = pr.get("body") or ""

    # GitHub uses null when a PR was never closed or merged.
    # We keep those values because they identify unmerged/open outcomes.
    pr["createdAt"] = pr.get("createdAt")
    pr["closedAt"] = pr.get("closedAt")
    pr["mergedAt"] = pr.get("mergedAt")

    pr["commits"] = pr.get("commits") or {"totalCount": 0}
    pr["reviews"] = pr.get("reviews") or {"totalCount": 0, "nodes": []}
    pr["comments"] = pr.get("comments") or {"totalCount": 0, "nodes": []}
    pr["reopened"] = pr.get("reopened") or {"totalCount": 0}

print("Missing-value handling complete: null dates are retained because they represent outcome status.")

Missing-value handling complete: null dates are retained because they represent outcome status.


**PART 3**

In [ ]:
# Compute metrics

for r in data:
    pr = r.get("pr")
    comments = pr.get("comments", {}).get("nodes", [])
    reviews = pr.get("reviews", {}).get("nodes", [])
    inline_comments = [comment for review in reviews for comment in review.get("comments", {}).get("nodes", [])]
    events = comments + reviews + inline_comments

    # Metric 1: number of comments and reviews
    r["n_comments"] = len(events)
    r["n_issue_comments"] = len(comments)
    r["n_review_comments"] = len(inline_comments)
    r["n_reviews"] = len(reviews)

    # Metric 2: number of participants
    reviewer_logins = {(review.get("author") or {}).get("login") for review in reviews if (review.get("author") or {}).get("login")}
    participant_logins = {(event.get("author") or {}).get("login") for event in events if (event.get("author") or {}).get("login")}
    r["n_reviewers"] = len(reviewer_logins)
    r["n_human_participants"] = len(participant_logins)

    # Metric 3: length of comments
    text = [event["n_words"] for event in events if event.get("n_words", 0) > 0]
    r["comment_words_median"] = (pd.Series(text).median() if text else 0)
    r["comment_words_mean"] = (pd.Series(text).mean() if text else 0)
    r["comment_words_total"] = sum(text)
    r["n_texted_comments"] = len(text)

    # Metric 4: response time
    created_times = [pd.to_datetime(event["createdAt"]) for event in events if event.get("createdAt")]
    if created_times:
        first_response = min(created_times)
        pr_created = pd.to_datetime(pr["createdAt"])
        r["first_response_at"] = first_response
        r["response_time_h"] = (first_response - pr_created).total_seconds() / 3600
    else:
        r["first_response_at"] = pd.NaT
        r["response_time_h"] = None

    # Metric 5: number of back and forth
    ordered_events = sorted(events, key=lambda e: e.get("createdAt") or "")
    sides = []

    for event in ordered_events:
        author = (event.get("author") or {}).get("login")
        pr_author = (pr.get("author") or {}).get("login")

        if bot_suffix.sub("", author).strip().lower() == bot_suffix.sub("", pr_author).strip().lower():
            sides.append("author")
        else:
            sides.append("other")

    sides = list(sides)
    r["n_alternations"] = sum(a != b for a, b in zip(sides, sides[1:]))
    
    # Metric 6: presence of clarification language
    r["has_clarification"] = any(event.get("is_clarification", False) for event in events)
    r["n_clarifications"] = sum(event.get("is_clarification", False) for event in events)

    # Metric 7: presence of approval#rejection language
    r["has_approval_lang"] = any(event.get("has_approval_lang", False) for event in events)
    r["has_rejection_lang"] = any(event.get("has_rejection_lang", False) for event in events)

    r["has_approved_review"] = any(review.get("state") == "APPROVED" for review in reviews)
    r["has_changes_requested"] = any(review.get("state") == "CHANGES_REQUESTED" for review in reviews)
    r["engaged"] = len(events) > 0

**METRICS FOR RQ1**

In [14]:
# Compute RQ1 metrics

for r in data:
    pr = r.get("pr") or {}

    reviews = pr.get("reviews", {}).get("nodes", [])

    for review in reviews:
        review.setdefault("createdAt", review.get("submittedAt"))

    comments = pr.get("comments", {}).get("nodes", [])
    inline_comments = [
        c for review in reviews
        for c in review.get("comments", {}).get("nodes", [])
    ]

    events = comments + reviews + inline_comments


    # Metric 1: merge outcome/rate
    r["merged"] = bool(pr.get("mergedAt"))
    r["closed_unmerged"] = bool(pr.get("closedAt")) and not r["merged"]


    # Metric 2: time to merge (hours)
    if pr.get("mergedAt") and pr.get("createdAt"):
        created = pd.to_datetime(pr["createdAt"], utc=True)
        merged = pd.to_datetime(pr["mergedAt"], utc=True)
        r["time_to_merge_h"] = (merged - created).total_seconds() / 3600
    else:
        r["time_to_merge_h"] = float("nan")


    # Metric 2b: time to close (only for PRs closed without merge)
    if r["closed_unmerged"] and pr.get("closedAt") and pr.get("createdAt"):
        created = pd.to_datetime(pr["createdAt"], utc=True)
        closed = pd.to_datetime(pr["closedAt"], utc=True)
        r["time_to_close_h"] = (closed - created).total_seconds() / 3600
    else:
        r["time_to_close_h"] = float("nan")


    # Metric 3: review rounds
    r["n_review_rounds"] = len(reviews)


    # Metric 4: commits before acceptance
    r["n_commits"] = pr.get("commits", {}).get("totalCount", 0) or 0


    # Metric 5: reopened/revised PR proxy
    r["n_reopens"] = pr.get("reopened", {}).get("totalCount", 0) or 0
    r["reopened"] = r["n_reopens"] > 0

    # Keep the existing collaboration variables because they are useful
    # for the same PR-level dataset and may support later RQs.
    r["n_comments"] = len(events)
    r["n_issue_comments"] = len(comments)
    r["n_review_comments"] = len(inline_comments)

    reviewer_logins = {
        (review.get("author") or {}).get("login")
        for review in reviews
        if (review.get("author") or {}).get("login")
    }
    participant_logins = {
        (event.get("author") or {}).get("login")
        for event in events
        if (event.get("author") or {}).get("login")
    }
    r["n_reviewers"] = len(reviewer_logins)
    r["n_human_participants"] = len(participant_logins)

    text_lengths = [event.get("n_words", 0) for event in events if event.get("n_words", 0) > 0]
    r["comment_words_median"] = pd.Series(text_lengths).median() if text_lengths else 0
    r["comment_words_mean"] = pd.Series(text_lengths).mean() if text_lengths else 0
    r["comment_words_total"] = sum(text_lengths)

    created_times = [
        pd.to_datetime(event["createdAt"], utc=True)
        for event in events
        if event.get("createdAt")
    ]
    if created_times and pr.get("createdAt"):
        first_response = min(created_times)
        pr_created = pd.to_datetime(pr["createdAt"], utc=True)
        r["first_response_at"] = first_response
        r["response_time_h"] = (first_response - pr_created).total_seconds() / 3600
    else:
        r["first_response_at"] = pd.NaT
        r["response_time_h"] = float("nan")

    ordered_events = sorted(events, key=lambda e: e.get("createdAt") or "")
    pr_author = (pr.get("author") or {}).get("login")
    sides = []
    for event in ordered_events:
        author = (event.get("author") or {}).get("login")
        sides.append("author" if author == pr_author else "other")
    r["n_alternations"] = sum(a != b for a, b in zip(sides, sides[1:]))

    r["has_clarification"] = any(event.get("is_clarification", False) for event in events)
    r["n_clarifications"] = sum(event.get("is_clarification", False) for event in events)
    r["has_approval_lang"] = any(event.get("has_approval_lang", False) for event in events)
    r["has_rejection_lang"] = any(event.get("has_rejection_lang", False) for event in events)
    r["has_approved_review"] = any(review.get("state") == "APPROVED" for review in reviews)
    r["has_changes_requested"] = any(review.get("state") == "CHANGES_REQUESTED" for review in reviews)
    r["engaged"] = len(events) > 0

print("RQ1 metrics computed for", len(data), "PRs")

RQ1 metrics computed for 10464 PRs


In [15]:
# Build and display the RQ1 comparison table (AI vs human)

rq1_df = pd.DataFrame([
    {
        "group": r["group"],
        "merged": r["merged"],
        "closed_unmerged": r["closed_unmerged"],
        "resolved": r["merged"] or r["closed_unmerged"],
        "time_to_merge_h": r["time_to_merge_h"],
        "time_to_close_h": r["time_to_close_h"],
        "n_review_rounds": r["n_review_rounds"],
        "commits_before_merge": r["n_commits"] if r["merged"] else float("nan"),
        "reopened": r["reopened"],
        "revised": r["has_changes_requested"],
    }
    for r in data
])

resolved = rq1_df[rq1_df["resolved"]]

from scipy import stats

def split(col):
    ai = resolved.loc[resolved["group"] == "ai", col].dropna()
    hu = resolved.loc[resolved["group"] == "human", col].dropna()
    return ai, hu

rows = []

for label, col in [
    ("Merge rate", "merged"),
    ("Issue closing rate (closed, not merged)", "closed_unmerged"),
    ("Reopened rate", "reopened"),
    ("Revised rate (changes requested)", "revised"),
]:
    ai, hu = split(col)
    ai, hu = ai.astype(int), hu.astype(int)
    total_pos, total_neg = ai.sum() + hu.sum(), (len(ai) - ai.sum()) + (len(hu) - hu.sum())
    p = float("nan") if total_pos == 0 or total_neg == 0 else stats.chi2_contingency(
        [[ai.sum(), len(ai) - ai.sum()], [hu.sum(), len(hu) - hu.sum()]]
    )[1]
    rows.append({"metric": label, "AI": f"{ai.mean():.1%}", "Human": f"{hu.mean():.1%}", "p_value": p})

fmt = lambda s: f"{s.median():.1f} [{s.quantile(.25):.1f}-{s.quantile(.75):.1f}]"
for label, col in [
    ("Time to merge (h)", "time_to_merge_h"),
    ("Time to close issue (h)", "time_to_close_h"),
    ("Review rounds", "n_review_rounds"),
    ("Commits before merge", "commits_before_merge"),
]:
    ai, hu = split(col)
    test = stats.mannwhitneyu(ai, hu, alternative="two-sided")
    rows.append({"metric": label, "AI": fmt(ai), "Human": fmt(hu), "p_value": test.pvalue})

rq1_summary = pd.DataFrame(rows).set_index("metric")
rq1_summary["p_value"] = rq1_summary["p_value"].map(lambda p: f"{p:.3g}" if pd.notna(p) else "n/a")
rq1_summary

,AI,Human,p_value
metric,,,
Merge rate,83.9%,82.4%,0.0431
"Issue closing rate (closed, not merged)",16.1%,17.6%,0.0431
Reopened rate,100.0%,100.0%,n/a
Revised rate (changes requested),0.3%,3.9%,5.79e-36
Time to merge (h),0.0 [0.0-0.2],4.4 [0.5-45.8],0
Time to close issue (h),7.0 [0.1-145.9],99.2 [3.1-889.1],7.2e-27
Review rounds,0.0 [0.0-0.0],1.0 [0.0-2.0],0
Commits before merge,1.0 [1.0-1.0],2.0 [1.0-4.0],0


In [16]:
# Transform the nested JSON into a flat table

rq1_rows = []

for r in data:
    pr = r.get("pr") or {}

    if pr.get("mergedAt"):
        outcome = "merged"
    elif pr.get("closedAt"):
        outcome = "closed_unmerged"
    else:
        outcome = "open"

    rq1_rows.append({
        "pr_id": r.get("pr_id"),
        "repo": r.get("repo"),
        "number": r.get("number"),
        "group": r.get("group"),          # "ai" or "human"
        "agent": r.get("agent"),
        "created_at": pr.get("createdAt"),
        "closed_at": pr.get("closedAt"),
        "merged_at": pr.get("mergedAt"),
        "outcome": outcome,                # merged / closed_unmerged / open
        "merged": r.get("merged"),
        "closed_unmerged": r.get("closed_unmerged"),
        "time_to_merge_h": r.get("time_to_merge_h"),
        "n_review_rounds": r.get("n_review_rounds"),
        "n_commits": r.get("n_commits"),
        "n_reopens": r.get("n_reopens"),
        "reopened": r.get("reopened"),
    })

rq1_df = pd.DataFrame(rq1_rows)

print(f"RQ1 dataset shape: {rq1_df.shape}")
print("\nOutcome counts by group (sanity check only):")
print(rq1_df.groupby("group")["outcome"].value_counts().to_string())

print("\nMissing values per RQ1 column (NaN in time_to_merge_h is expected for non-merged PRs):")
print(rq1_df.isna().sum().to_string())

RQ1 dataset shape: (10464, 16)

Outcome counts by group (sanity check only):
group  outcome        
ai     merged             4428
       closed_unmerged     851
       open                 83
human  merged             4134
       closed_unmerged     885
       open                 83

Missing values per RQ1 column (NaN in time_to_merge_h is expected for non-merged PRs):
pr_id                 0
repo                  0
number                0
group                 0
agent                 0
created_at            0
closed_at           166
merged_at          1902
outcome               0
merged                0
closed_unmerged       0
time_to_merge_h    1902
n_review_rounds       0
n_commits             0
n_reopens             0
reopened              0


In [17]:
# Save the RQ1 metrics table in CSV.
import csv

rq1_output_path = data_dir / "rq1_metrics.csv"

with rq1_output_path.open("w", newline="", encoding="utf-8-sig") as f:
    f.write("sep=,\n")  
    rq1_df.to_csv(f, index=False, quoting=csv.QUOTE_ALL)

print(f"Saved RQ1 metrics table to: {rq1_output_path}")

Saved RQ1 metrics table to: /Users/randy.rakotondrazafy/LOG6307E Data Minig/Replication_Project_LOG6307E/data/rq1_metrics.csv
